# YOLOv8n Training — Denim Fabric Defect Detection

This notebook reproduces the model-training pipeline used in the thesis
*"A Closed-Loop YOLOv8 System for Real-Time Denim Defect Detection and Control."*
It is written for **Google Colab with a GPU runtime** (Runtime → Change runtime type → GPU).

Stages: environment/GPU check → dataset extraction → `data.yaml` config →
YOLOv8n training (100 epochs) → save to Drive → validation at two resolutions →
inference-speed (FPS) benchmark.

> **Paths:** the cells use Google Drive paths from the original experiments
> (e.g. `/content/drive/MyDrive/fabric/...`). Edit them to match your own Drive layout.


## B.1 — Environment and GPU verification

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "NONE - check Runtime type!")

In [ ]:
!pip install ultralytics -q

## B.2 — Dataset extraction and verification

The dataset is stored as a ZIP on Google Drive, extracted to the Colab runtime. Class IDs in the label files are verified before training.

In [ ]:
import zipfile, os, glob

zip_path = '/content/drive/MyDrive/fabric/dataset_clean.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/data')

# Confirm extracted structure
for root, dirs, files in os.walk('/content/data'):
    if files:
        print(root, '->', len(files), 'files')

# Verify class IDs in label files
ids = set()
for lbl in glob.glob('/content/data/**/labels/**/*.txt', recursive=True):
    for line in open(lbl):
        if line.strip():
            ids.add(int(line.split()[0]))
print("Class IDs present in labels:", sorted(ids))

## B.3 — Dataset configuration (`data.yaml`)

Defines the dataset root, train/val paths, and the four defect class names.

In [ ]:
yaml_content = """
path: /content/data/dataset_clean
train: images/train
val: images/val

names:
  0: Holes
  1: Abrasion_Mark
  2: Oil_Stain
  3: Missing_Yarn
"""

with open('/content/data.yaml', 'w') as f:
    f.write(yaml_content)

print(open('/content/data.yaml').read())

## B.4 — Model training

Fine-tunes YOLOv8n for 100 epochs at 768x768. AdamW optimiser, cosine LR decay, built-in mosaic/mixup augmentation, early stopping (patience 20).

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # pretrained YOLOv8 Nano backbone

results = model.train(
    data       = "/content/data.yaml",
    epochs     = 100,
    imgsz      = 768,          # high resolution for small defects
    batch      = 16,
    device     = 0,            # GPU
    workers    = 8,
    patience   = 20,           # early stopping
    pretrained = True,
    optimizer  = "AdamW",
    lr0        = 0.001,
    cos_lr     = True,         # cosine LR schedule
    augment    = True,
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    flipud     = 0.0,          # no vertical flip (preserves fabric orientation)
    fliplr     = 0.5,
    mosaic     = 1.0,
    mixup      = 0.1,
    box        = 7.5,
    cls        = 0.5,
    dfl        = 1.5,
    name       = "fabric_defect_yolov8n_clean",
    exist_ok   = True,
)

## B.5 — Save results to Google Drive

Copies trained weights and logs back to Drive for persistence.

In [ ]:
import shutil

shutil.copytree(
    '/content/runs',
    '/content/drive/MyDrive/fabric/runs_clean',
    dirs_exist_ok=True
)
print("Saved to Drive: fabric/runs_clean")

## B.6 — Validation and per-class metrics

Locates the best checkpoint and validates on the held-out set at **768** and **640** (the deployment resolution). Reports mAP@0.5, mAP@0.5:0.95, and per-class AP@0.5.

In [ ]:
import glob, yaml
from ultralytics import YOLO

# Locate best checkpoint
best_pt = glob.glob(
    '/content/drive/MyDrive/fabric/runs_clean/**/best.pt',
    recursive=True
)[0]
m = YOLO(best_pt)
print("Class names from model:", m.names)

# Rebuild data.yaml using model's own class order
data_cfg = {
    'path' : '/content/dataset/dataset_clean',
    'train': 'images/train',
    'val'  : 'images/val',
    'names': {int(k): v for k, v in m.names.items()},
}
with open('/content/data.yaml', 'w') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

# Validate at two resolutions
DATA = "/content/data.yaml"
for res in (768, 640):
    print(f"\n===== Validation @ {res}x{res} =====")
    r = m.val(data=DATA, imgsz=res, split="val", verbose=False)
    print(f"mAP@0.5      : {r.box.map50:.4f}")
    print(f"mAP@0.5:0.95 : {r.box.map:.4f}")
    for i, ci in enumerate(r.box.ap_class_index):
        print(f"  {m.names[int(ci)]:<20} AP@0.5 = {r.box.ap50[i]:.4f}")

## B.7 — Inference-speed (FPS) benchmark

Measures per-frame preprocess/inference/postprocess time at 640x640 after a 20-frame GPU warm-up. Reports mean +/- std (ms) and FPS.

In [ ]:
import numpy as np, glob
from ultralytics import YOLO

model = YOLO(best_pt)
imgs = sorted(glob.glob("/content/dataset/dataset_clean/images/val/*.jpg"))
print(f"{len(imgs)} validation images found")

# GPU warm-up (first runs are artificially slow)
for _ in range(20):
    model.predict(imgs[0], imgsz=640, device=0, verbose=False)

pre, inf, post = [], [], []
for img in imgs:
    s = model.predict(img, imgsz=640, device=0, verbose=False)[0].speed
    pre.append(s['preprocess'])
    inf.append(s['inference'])
    post.append(s['postprocess'])

pre, inf, post = map(np.array, (pre, inf, post))
total = pre + inf + post

print(f"Preprocessing  : {pre.mean():.2f} +/- {pre.std():.2f} ms")
print(f"Inference      : {inf.mean():.2f} +/- {inf.std():.2f} ms")
print(f"Postprocessing : {post.mean():.2f} +/- {post.std():.2f} ms")
print(f"Total / frame  : {total.mean():.2f} +/- {total.std():.2f} ms")
print(f"Inference FPS  : {1000/inf.mean():.1f}")
print(f"End-to-end FPS : {1000/total.mean():.1f}")